In [ ]:
import json
import os
from datetime import datetime

GOREV_DOSYASI = "gorevler.json"
ONCELIK_DEGERLERI = {"Düşük": 1, "Orta": 2, "Yüksek": 3}


In [ ]:
def gorevleri_oku():
    if os.path.exists(GOREV_DOSYASI):
        try:
            with open(GOREV_DOSYASI, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception as e:
            print("Dosya okuma hatası:", e)
            return []
    return []


In [ ]:
def gorevleri_kaydet(gorevler):
    try:
        with open(GOREV_DOSYASI, "w", encoding="utf-8") as f:
            json.dump(gorevler, f, ensure_ascii=False, indent=4)
    except Exception as e:
        print("Dosya yazma hatası:", e)


In [ ]:
def gorevleri_listele(gorevler, siralama_kriteri=None):
    if not gorevler:
        print("Görev listesi boş.")
        return

    if siralama_kriteri == "öncelik":
        gorevler = sorted(gorevler, key=lambda x: ONCELIK_DEGERLERI.get(x["öncelik"], 0), reverse=True)
    elif siralama_kriteri == "tarih":
        def tarih_key(g):
            try:
                return datetime.strptime(g["son_tarih"], "%Y-%m-%d")
            except:
                return datetime.max
        gorevler = sorted(gorevler, key=tarih_key)
    elif siralama_kriteri == "tamamlanma":
        gorevler = sorted(gorevler, key=lambda x: x["tamamlandi"])

    print("\n--- GÖREV LİSTESİ ---")
    for i, g in enumerate(gorevler, 1):
        durum = "✓" if g["tamamlandi"] else "✗"
        tarih = g["son_tarih"] if g["son_tarih"] else "Yok"
        print(f"{i}. [{durum}] {g['metin']} (Öncelik: {g['öncelik']}, Son Tarih: {tarih})")


In [ ]:
def yeni_gorev_ekle(gorevler):
    metin = input("Görev metnini girin: ").strip()
    if not metin:
        print("Boş görev eklenemez!")
        return

    print("Öncelik seviyesini seçin: (Düşük / Orta / Yüksek)")
    oncelik = input().strip().capitalize()
    if oncelik not in ONCELIK_DEGERLERI:
        print("Geçersiz öncelik, varsayılan 'Orta' seçildi.")
        oncelik = "Orta"

    son_tarih = input("Son tarih (YYYY-AA-GG formatında, boş bırakabilirsiniz): ").strip()
    if son_tarih:
        try:
            datetime.strptime(son_tarih, "%Y-%m-%d")
        except ValueError:
            print("Geçersiz tarih formatı, tarih eklenmedi.")
            son_tarih = ""
    else:
        son_tarih = ""

    yeni_gorev = {
        "metin": metin,
        "tamamlandi": False,
        "öncelik": oncelik,
        "son_tarih": son_tarih
    }
    gorevler.append(yeni_gorev)
    gorevleri_kaydet(gorevler)
    print("Görev eklendi.")


In [ ]:
def gorev_sil(gorevler):
    gorevleri_listele(gorevler)
    try:
        numara = int(input("Silmek istediğiniz görev numarası: "))
        if 1 <= numara <= len(gorevler):
            silinen = gorevler.pop(numara - 1)
            gorevleri_kaydet(gorevler)
            print(f"'{silinen['metin']}' silindi.")
        else:
            print("Geçersiz görev numarası!")
    except ValueError:
        print("Lütfen geçerli bir sayı girin.")


In [ ]:
def gorev_duzenle(gorevler):
    gorevleri_listele(gorevler)
    try:
        numara = int(input("Düzenlemek istediğiniz görev numarası: "))
        if 1 <= numara <= len(gorevler):
            g = gorevler[numara - 1]
            yeni_metin = input(f"Yeni görev metni (Mevcut: {g['metin']}): ").strip()
            if yeni_metin:
                g["metin"] = yeni_metin
            else:
                print("Görev metni boş bırakılamaz!")

            print(f"Öncelik (Düşük / Orta / Yüksek), mevcut: {g['öncelik']}")
            yeni_oncelik = input("Yeni öncelik (boş bırakılırsa değiştirilmez): ").strip().capitalize()
            if yeni_oncelik:
                if yeni_oncelik in ONCELIK_DEGERLERI:
                    g["öncelik"] = yeni_oncelik
                else:
                    print("Geçersiz öncelik, değiştirilmedi.")

            print(f"Son tarih (YYYY-AA-GG), mevcut: {g['son_tarih'] if g['son_tarih'] else 'Yok'}")
            yeni_tarih = input("Yeni son tarih (boş bırakılırsa değiştirilmez): ").strip()
            if yeni_tarih:
                try:
                    datetime.strptime(yeni_tarih, "%Y-%m-%d")
                    g["son_tarih"] = yeni_tarih
                except ValueError:
                    print("Geçersiz tarih formatı, değiştirilmedi.")

            gorevleri_kaydet(gorevler)
            print("Görev güncellendi.")
        else:
            print("Geçersiz görev numarası!")
    except ValueError:
        print("Lütfen geçerli bir sayı girin.")


In [ ]:
def gorev_tamamla(gorevler):
    gorevleri_listele(gorevler)
    try:
        numara = int(input("Tamamlandı olarak işaretlemek istediğiniz görev numarası: "))
        if 1 <= numara <= len(gorevler):
            g = gorevler[numara - 1]
            g["tamamlandi"] = True
            gorevleri_kaydet(gorevler)
            print(f"'{g['metin']}' tamamlandı olarak işaretlendi.")
        else:
            print("Geçersiz görev numarası!")
    except ValueError:
        print("Lütfen geçerli bir sayı girin.")


In [ ]:
def menu():
    gorevler = gorevleri_oku()
    while True:
        print("\n--- GÖREV YÖNETİM UYGULAMASI ---")
        print("1. Görevleri Listele")
        print("2. Yeni Görev Ekle")
        print("3. Görev Düzenle")
        print("4. Görev Sil")
        print("5. Görev Tamamla")
        print("6. Görevleri Önceliğe Göre Sırala")
        print("7. Görevleri Son Tarihe Göre Sırala")
        print("8. Görevleri Tamamlanma Durumuna Göre Sırala")
        print("9. Çıkış")

        secim = input("Seçiminiz (1-9): ")

        if secim == "1":
            gorevleri_listele(gorevler)
        elif secim == "2":
            yeni_gorev_ekle(gorevler)
        elif secim == "3":
            gorev_duzenle(gorevler)
        elif secim == "4":
            gorev_sil(gorevler)
        elif secim == "5":
            gorev_tamamla(gorevler)
        elif secim == "6":
            gorevleri_listele(gorevler, siralama_kriteri="öncelik")
        elif secim == "7":
            gorevleri_listele(gorevler, siralama_kriteri="tarih")
        elif secim == "8":
            gorevleri_listele(gorevler, siralama_kriteri="tamamlanma")
        elif secim == "9":
            print("Çıkılıyor...")
            break
        else:
            print("Geçersiz seçim. Lütfen 1-9 arasında bir değer girin.")


In [ ]:
menu()



--- GÖREV YÖNETİM UYGULAMASI ---
1. Görevleri Listele
2. Yeni Görev Ekle
3. Görev Düzenle
4. Görev Sil
5. Görev Tamamla
6. Görevleri Önceliğe Göre Sırala
7. Görevleri Son Tarihe Göre Sırala
8. Görevleri Tamamlanma Durumuna Göre Sırala
9. Çıkış
Seçiminiz (1-9): 1
Görev listesi boş.

--- GÖREV YÖNETİM UYGULAMASI ---
1. Görevleri Listele
2. Yeni Görev Ekle
3. Görev Düzenle
4. Görev Sil
5. Görev Tamamla
6. Görevleri Önceliğe Göre Sırala
7. Görevleri Son Tarihe Göre Sırala
8. Görevleri Tamamlanma Durumuna Göre Sırala
9. Çıkış
Seçiminiz (1-9): 9
Çıkılıyor...
